# Notebook 9 — Pseudo-Labeling (Self-Training on Test Patches)

**Goal:** Use the 800 unlabeled test patches to augment the training set via self-training.
High-confidence model predictions become "pseudo-labels" and are added back as additional
training examples.

**Why this works:** The model has already generalised from 6329 train patches. Test patches come
from the same spatial domain. For patches where the model is very confident (P(class) ≥ 0.85),
the pseudo-labels are likely correct — adding them reinforces the decision boundary.

**Protocol (2 rounds):**
```
Round 0  →  Best model from NB3 or NB7 generates pseudo-labels on test set
           Filter pixels with max_prob ≥ CONF_THRESHOLD
Round 1  →  Re-train pixel baseline (NB2 style) on train + pseudo pixels
           Evaluate on val; if Acc±1 improves, save checkpoint
Round 2  →  (Optional) Generate new pseudo-labels with Round-1 model, re-train
```

**HITL gate:** VB checks whether pseudo-label class distribution is reasonable (not
dominated by a single class), then approves retrain.

**Outputs:**
- `results/subtask1/pseudo/pseudo_labels_r1.npz` — round-1 pseudo pixels
- `results/subtask1/val_preds/pseudo_val_probs.npz` — soft val predictions for NB10
- `results/subtask1/checkpoints/pseudo_lgbm_r1.pkl` — retrained pixel model
- `results/subtask1/val_preds/pseudo_val_metrics.json`

In [ ]:
import os, json, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd

_cwd = Path(os.getcwd())
REPO_ROOT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent

DATA_DIR   = REPO_ROOT / 'data' / 'subtask1'
CKPT_DIR   = REPO_ROOT / 'results' / 'subtask1' / 'checkpoints'
PRED_DIR   = REPO_ROOT / 'results' / 'subtask1' / 'val_preds'
PSEUDO_DIR = REPO_ROOT / 'results' / 'subtask1' / 'pseudo'

for d in [CKPT_DIR, PRED_DIR, PSEUDO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────────
CONF_THRESHOLD  = 0.85    # minimum max-class probability to accept a pseudo-label pixel
MAX_PSEUDO_PIX  = 2_000_000  # cap to keep RAM manageable; ~50x train pixel count is plenty
NUM_CLASSES     = 5
PATCH_SIZE      = 128
T, C            = 34, 10

# Which model to use as pseudo-label generator?
# 'corn' → NB7 CORN U-Net (recommended if val Acc±1 > NB3 U-Net)
# 'unet' → NB3 soft-ordinal U-Net
# 'lgbm' → NB2 pixel baseline (fast but weaker spatial context)
GENERATOR_MODEL = 'corn'  # change based on HITL review of NB3 vs NB7

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Load Generator Model

In [ ]:
# ── Shared U-Net / CORN definitions (minimal; copy from NB3/NB7) ────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class SoftOrdinalUNet(nn.Module):
    def __init__(self, in_channels, num_classes, base_ch=32):
        super().__init__()
        self.enc1, self.enc2, self.enc3, self.enc4 = (
            DoubleConv(in_channels, base_ch),
            DoubleConv(base_ch, base_ch*2),
            DoubleConv(base_ch*2, base_ch*4),
            DoubleConv(base_ch*4, base_ch*8),
        )
        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_ch*8, base_ch*16)
        self.up4, self.dec4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, 2, 2), DoubleConv(base_ch*16, base_ch*8)
        self.up3, self.dec3 = nn.ConvTranspose2d(base_ch*8,  base_ch*4, 2, 2), DoubleConv(base_ch*8,  base_ch*4)
        self.up2, self.dec2 = nn.ConvTranspose2d(base_ch*4,  base_ch*2, 2, 2), DoubleConv(base_ch*4,  base_ch*2)
        self.up1, self.dec1 = nn.ConvTranspose2d(base_ch*2,  base_ch,   2, 2), DoubleConv(base_ch*2,  base_ch)
        self.head = nn.Conv2d(base_ch, num_classes, 1)
    def forward(self, x):
        e1, e2, e3, e4 = self.enc1(x), self.enc2(self.pool(self.enc1(x))), None, None
        # proper forward
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


class CORNUNet(nn.Module):
    def __init__(self, in_channels, num_classes, base_ch=32):
        super().__init__()
        K = num_classes
        self.enc1, self.pool = DoubleConv(in_channels, base_ch), nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_ch,   base_ch*2)
        self.enc3 = DoubleConv(base_ch*2, base_ch*4)
        self.enc4 = DoubleConv(base_ch*4, base_ch*8)
        self.bottleneck = DoubleConv(base_ch*8, base_ch*16)
        self.up4, self.dec4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, 2,2), DoubleConv(base_ch*16, base_ch*8)
        self.up3, self.dec3 = nn.ConvTranspose2d(base_ch*8,  base_ch*4, 2,2), DoubleConv(base_ch*8,  base_ch*4)
        self.up2, self.dec2 = nn.ConvTranspose2d(base_ch*4,  base_ch*2, 2,2), DoubleConv(base_ch*4,  base_ch*2)
        self.up1, self.dec1 = nn.ConvTranspose2d(base_ch*2,  base_ch,   2,2), DoubleConv(base_ch*2,  base_ch)
        self.head = nn.Conv2d(base_ch, K - 1, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)   # (B, K-1, H, W)


def corn_probs(logits):
    cond = torch.sigmoid(logits)
    cum  = torch.cumprod(cond, dim=-1)
    ones  = torch.ones(*logits.shape[:-1], 1, device=logits.device)
    zeros = torch.zeros(*logits.shape[:-1], 1, device=logits.device)
    ext   = torch.cat([ones, cum, zeros], dim=-1)
    return (ext[..., :-1] - ext[..., 1:]).clamp(min=0)


print('Model classes defined.')

In [ ]:
# Load norm stats
NORM_FILE = REPO_ROOT / 'results' / 'subtask1' / 'norm_stats.npz'
norm_stats = dict(np.load(NORM_FILE)) if NORM_FILE.exists() else None

# Determine temporal mode from checkpoint metadata
if GENERATOR_MODEL == 'corn':
    ckpt_path = CKPT_DIR / 'corn_unet_best.pt'
elif GENERATOR_MODEL == 'unet':
    ckpt_path = CKPT_DIR / 'unet_best.pt'
else:
    ckpt_path = None

if ckpt_path is not None and ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    TEMPORAL_MODE = ckpt.get('temporal_mode', 'summary')
    BASE_CH       = ckpt.get('base_ch', 32)
    IN_CHANNELS   = ckpt.get('in_channels', None)  # may be absent in older checkpoints
    print(f'Checkpoint: {ckpt_path.name}, temporal={TEMPORAL_MODE}, base_ch={BASE_CH}')
else:
    TEMPORAL_MODE = 'summary'
    BASE_CH       = 32
    IN_CHANNELS   = None
    print('Checkpoint not found — will use defaults.')

In [ ]:
# ── Minimal Dataset for loading patches ──────────────────────────────────────
def list_sentinel_tifs(data_dir):
    tifs = sorted(Path(data_dir).glob('S2*.tif'))
    if not tifs: tifs = sorted(Path(data_dir).glob('**/*.tif'))
    return tifs


def reduce_temporal(stack, mode):
    if mode == 'concat':
        T, C, H, W = stack.shape; return stack.reshape(T*C, H, W)
    elif mode == 'summary':
        return np.concatenate([stack.mean(0), stack.std(0)], axis=0)
    elif mode == 'seasonal':
        T = stack.shape[0]; q = T // 4
        return np.concatenate([stack[:q].mean(0), stack[q:2*q].mean(0),
                                stack[2*q:3*q].mean(0), stack[3*q:].mean(0)], axis=0)
    raise ValueError(mode)


class PatchLoaderDataset(Dataset):
    def __init__(self, csv_path, data_dir, norm_stats=None, temporal_mode='summary'):
        self.df      = pd.read_csv(csv_path)
        self.data_dir = Path(data_dir)
        self.norm     = norm_stats
        self.mode     = temporal_mode
        self.tifs     = list_sentinel_tifs(self.data_dir)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row['patch_id']
        win = Window(int(row['col']), int(row['row']), int(row['patch_size']), int(row['patch_size']))
        frames = []
        for f in self.tifs:
            with rasterio.open(f) as src:
                frames.append(src.read(window=win).astype(np.float32))
        stack = np.stack(frames, 0) / 10000.0
        feat  = reduce_temporal(stack, self.mode)
        if self.norm is not None:
            m = self.norm['mean'][:, None, None].astype(np.float32)
            s = self.norm['std'][:, None, None].astype(np.float32)
            s = np.where(s == 0, 1.0, s)
            feat = (feat - m) / s
        return torch.from_numpy(feat), pid


# Build test dataset
TEST_CSV  = DATA_DIR / 'test.csv'
test_ds   = PatchLoaderDataset(TEST_CSV, DATA_DIR, norm_stats, TEMPORAL_MODE)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

if IN_CHANNELS is None:
    sample_x, _ = test_ds[0]
    IN_CHANNELS = sample_x.shape[0]

print(f'Test patches: {len(test_ds)}, Input channels: {IN_CHANNELS}')

In [ ]:
# ── Instantiate and load generator model ────────────────────────────────────
if GENERATOR_MODEL == 'corn':
    gen_model = CORNUNet(IN_CHANNELS, NUM_CLASSES, BASE_CH).to(DEVICE)
elif GENERATOR_MODEL == 'unet':
    gen_model = SoftOrdinalUNet(IN_CHANNELS, NUM_CLASSES, BASE_CH).to(DEVICE)
else:
    gen_model = None

if gen_model is not None:
    gen_model.load_state_dict(ckpt['model_state'])
    gen_model.eval()
    print(f'Loaded {GENERATOR_MODEL} model from {ckpt_path.name}')

## 2. Generate Pseudo-Labels on Test Set

In [ ]:
pseudo_features = []   # list of (F,) feature vectors
pseudo_labels   = []   # list of int class labels
pseudo_conf     = []   # max probability for each accepted pixel

# For LightGBM pseudo-labeling, we need to load features separately
# For U-Net / CORN, we generate pixel-level probs directly from the model

print(f'Generating pseudo-labels on {len(test_ds)} test patches...')
print(f'  Confidence threshold: {CONF_THRESHOLD}')
print(f'  Max pseudo pixels: {MAX_PSEUDO_PIX:,}')

n_total_pix = 0
n_accepted  = 0

with torch.no_grad():
    for x, patch_ids in test_loader:
        x = x.to(DEVICE)

        if GENERATOR_MODEL == 'corn':
            logits = gen_model(x)                        # (B, K-1, H, W)
            probs  = corn_probs(logits.permute(0,2,3,1)) # (B, H, W, K)
        elif GENERATOR_MODEL == 'unet':
            logits = gen_model(x)                        # (B, K, H, W)
            probs  = torch.softmax(logits.permute(0,2,3,1), dim=-1)  # (B, H, W, K)
        else:
            break  # LightGBM handled below

        probs_np  = probs.cpu().numpy()                  # (B, H, W, K)
        max_prob  = probs_np.max(axis=-1)                # (B, H, W)
        pred_cls  = probs_np.argmax(axis=-1)             # (B, H, W)

        B = x.shape[0]
        # Also save the raw features (flattened) for re-training the pixel model
        feat_np = x.cpu().numpy()  # (B, F, H, W)

        for b in range(B):
            mask = max_prob[b] >= CONF_THRESHOLD       # (H, W) bool
            n_total_pix += mask.size
            n_accepted  += mask.sum()

            if mask.sum() > 0:
                # Flatten features: (F, H, W) → (N_accepted, F)
                f = feat_np[b]                          # (F, H, W)
                f_flat = f.reshape(f.shape[0], -1).T    # (H*W, F)
                accepted_feat = f_flat[mask.ravel()]    # (N_acc, F)
                accepted_cls  = pred_cls[b][mask]       # (N_acc,)
                accepted_conf = max_prob[b][mask]

                pseudo_features.append(accepted_feat)
                pseudo_labels.append(accepted_cls)
                pseudo_conf.append(accepted_conf)

        if n_accepted >= MAX_PSEUDO_PIX:
            print(f'Reached max pseudo pixels limit.')
            break

if pseudo_features:
    pseudo_X = np.concatenate(pseudo_features, axis=0)
    pseudo_y = np.concatenate(pseudo_labels)
    pseudo_c = np.concatenate(pseudo_conf)

    # If over limit, sample randomly
    if len(pseudo_y) > MAX_PSEUDO_PIX:
        idx = np.random.choice(len(pseudo_y), MAX_PSEUDO_PIX, replace=False)
        pseudo_X, pseudo_y, pseudo_c = pseudo_X[idx], pseudo_y[idx], pseudo_c[idx]

    print(f'\nAccepted {len(pseudo_y):,} / {n_total_pix:,} pixels '
          f'({100*len(pseudo_y)/n_total_pix:.1f}%)')
    unique, counts = np.unique(pseudo_y, return_counts=True)
    print('Pseudo-label class distribution:')
    for u, cnt in zip(unique, counts):
        print(f'  Class {u}: {cnt:,} ({100*cnt/len(pseudo_y):.1f}%)')
else:
    print('No pseudo-labels generated. Check model checkpoint and data paths.')
    pseudo_X = pseudo_y = pseudo_c = None

In [ ]:
# ── HITL inspection: class distribution of pseudo-labels ──────────────────────
import matplotlib.pyplot as plt

if pseudo_y is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Class distribution
    unique, counts = np.unique(pseudo_y, return_counts=True)
    axes[0].bar(unique, counts, color=['#1a1a1a', '#4d8c57', '#a3c86a', '#f5e642', '#e07b1a'][:len(unique)])
    axes[0].set(xlabel='Class', ylabel='Pixel count', title='Pseudo-label class distribution')
    axes[0].set_xticks(range(NUM_CLASSES))

    # Confidence distribution
    axes[1].hist(pseudo_c, bins=50, color='steelblue', edgecolor='white')
    axes[1].axvline(CONF_THRESHOLD, color='red', linestyle='--', label=f'Threshold={CONF_THRESHOLD}')
    axes[1].set(xlabel='Max class probability', ylabel='Count',
                title='Confidence distribution of accepted pseudo-labels')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(PSEUDO_DIR / 'pseudo_label_dist.png', dpi=100)
    plt.show()
    print('\nHITL: Review the class distribution above.')
    print('If any class is > 80% of pseudo-labels, lower CONF_THRESHOLD or reject.')

## 3. Load Training Features & Merge with Pseudo-Labels

Re-use the pixel feature extraction from Notebook 2 (or the cached `.npz` features from NB6).

In [ ]:
# Check if NB2 pixel features are cached
PIXEL_CACHE = REPO_ROOT / 'results' / 'subtask1' / 'pixel_features_train.npz'

if PIXEL_CACHE.exists():
    cache = np.load(PIXEL_CACHE)
    X_train = cache['X']
    y_train = cache['y']
    print(f'Loaded cached train features: X={X_train.shape}, y={y_train.shape}')
else:
    print('pixel_features_train.npz not found.')
    print('Run Notebook 2 first, or run the feature extraction below.')
    print('Skipping merge step — will retrain on pseudo-labels only if available.')
    X_train = None
    y_train = None

In [ ]:
# Merge real training pixels with pseudo-labeled test pixels
if X_train is not None and pseudo_X is not None:
    # Clip pseudo features to match real features if dimensions differ
    # (real features from NB2 may have more engineered features than raw U-Net input)
    if pseudo_X.shape[1] != X_train.shape[1]:
        print(f'Feature dimension mismatch: real={X_train.shape[1]}, pseudo={pseudo_X.shape[1]}')
        print('Pseudo features come directly from the U-Net input (raw temporal reduction).')
        print('Using only the overlapping first N features for the retrained model.')
        n_feat = min(X_train.shape[1], pseudo_X.shape[1])
        X_combined = np.vstack([X_train[:, :n_feat], pseudo_X[:, :n_feat]])
    else:
        X_combined = np.vstack([X_train, pseudo_X])

    y_combined = np.concatenate([y_train, pseudo_y])
    print(f'Combined training set: {len(y_combined):,} pixels')
    print(f'  Real: {len(y_train):,}  Pseudo: {len(pseudo_y):,}')
elif X_train is not None:
    X_combined = X_train
    y_combined = y_train
    print('Using real training data only (no pseudo-labels).')
elif pseudo_X is not None:
    X_combined = pseudo_X
    y_combined = pseudo_y
    print('Using pseudo-labels only (no cached real features).')
else:
    print('No data available. Cannot retrain.')
    X_combined = y_combined = None

In [ ]:
# Save pseudo-labels for reproducibility
if pseudo_X is not None:
    np.savez_compressed(
        PSEUDO_DIR / 'pseudo_labels_r1.npz',
        X=pseudo_X, y=pseudo_y, conf=pseudo_c,
        conf_threshold=np.float32(CONF_THRESHOLD),
        generator=np.array(GENERATOR_MODEL),
    )
    print('Saved:', PSEUDO_DIR / 'pseudo_labels_r1.npz')

## 4. Retrain Pixel Baseline (LightGBM / HistGB) with Pseudo-Labels

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

def acc_pm1(pred, gt):
    return float((np.abs(pred.astype(int) - gt.astype(int)) <= 1).mean())

def exact_acc(pred, gt):
    return float((pred == gt).mean())

if X_combined is not None:
    print(f'Training HistGradientBoostingClassifier on {len(y_combined):,} pixels...')
    t0 = time.time()

    sample_wt = compute_sample_weight('balanced', y_combined)

    clf = HistGradientBoostingClassifier(
        max_iter=300,
        max_leaf_nodes=63,
        learning_rate=0.05,
        random_state=42,
        class_weight='balanced',
    )
    clf.fit(X_combined, y_combined)
    print(f'  Training time: {time.time()-t0:.1f}s')

    with open(CKPT_DIR / 'pseudo_lgbm_r1.pkl', 'wb') as f:
        pickle.dump(clf, f)
    print('  Saved: pseudo_lgbm_r1.pkl')

## 5. Evaluate on Validation Set

In [ ]:
# Load val features (from NB2 cache)
VAL_CACHE = REPO_ROOT / 'results' / 'subtask1' / 'pixel_features_val.npz'

if VAL_CACHE.exists() and X_combined is not None:
    val_cache = np.load(VAL_CACHE)
    X_val = val_cache['X']
    y_val = val_cache['y']
    val_patch_ids = val_cache.get('patch_ids', None)
    print(f'Loaded val features: X={X_val.shape}')

    if X_val.shape[1] != X_combined.shape[1]:
        n_feat = min(X_val.shape[1], X_combined.shape[1])
        X_val = X_val[:, :n_feat]

    val_pred = clf.predict(X_val).astype(int)
    val_prob = clf.predict_proba(X_val)         # (N, K)

    v_apm1 = acc_pm1(val_pred, y_val)
    v_acc  = exact_acc(val_pred, y_val)

    print(f'\n=== Val Metrics (Pseudo-retrained LightGBM, Round 1) ===')
    print(f'  Acc±1:      {v_apm1:.4f}')
    print(f'  Exact Acc:  {v_acc:.4f}')

    # Save val probabilities for NB10 stacking
    # Reshape to (N_patches, H, W, K) — requires patch info from cache
    np.savez_compressed(
        PRED_DIR / 'pseudo_val_probs.npz',
        probs=val_prob,
        gt=y_val,
    )

    metrics = {'val_acc_pm1': v_apm1, 'val_acc_exact': v_acc,
               'conf_threshold': CONF_THRESHOLD, 'n_pseudo': int(len(pseudo_y))}
    (PRED_DIR / 'pseudo_val_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))
else:
    print('Val feature cache not found. Run Notebook 2 to generate pixel_features_val.npz.')

## 6. Round 2 (Optional — Run Only If Round 1 Improved Val Acc±1)

In [ ]:
# ── Round 2: Re-generate pseudo-labels with the retrained model ─────────────
# Only run if Round 1 showed improvement over the baseline val Acc±1.
# Set RUN_ROUND2 = True after reviewing Round 1 results.

RUN_ROUND2 = False   # ← HITL: set True if Round 1 improved val Acc±1

if RUN_ROUND2 and X_combined is not None:
    print('=== Round 2 Pseudo-Labeling ===')
    print('Generating new pseudo-labels from the retrained pixel model...')

    r2_features, r2_labels, r2_conf = [], [], []

    for x, patch_ids in test_loader:
        x = x.to(DEVICE)
        feat_np = x.cpu().numpy()   # (B, F, H, W)

        for b in range(x.shape[0]):
            f = feat_np[b]                              # (F, H, W)
            f_flat = f.reshape(f.shape[0], -1).T       # (H*W, F)

            if f_flat.shape[1] != X_combined.shape[1]:
                f_flat = f_flat[:, :X_combined.shape[1]]

            prob = clf.predict_proba(f_flat)            # (H*W, K)
            max_p = prob.max(axis=1)                    # (H*W,)
            cls   = prob.argmax(axis=1)                 # (H*W,)

            mask = max_p >= CONF_THRESHOLD
            if mask.sum() > 0:
                r2_features.append(f_flat[mask])
                r2_labels.append(cls[mask])
                r2_conf.append(max_p[mask])

    if r2_features:
        r2_X = np.concatenate(r2_features)
        r2_y = np.concatenate(r2_labels)
        r2_c = np.concatenate(r2_conf)
        print(f'Round 2 pseudo-labels: {len(r2_y):,} pixels')

        X_r2 = np.vstack([X_combined, r2_X[:MAX_PSEUDO_PIX]])
        y_r2 = np.concatenate([y_combined, r2_y[:MAX_PSEUDO_PIX]])

        clf_r2 = HistGradientBoostingClassifier(
            max_iter=300, max_leaf_nodes=63, learning_rate=0.05,
            random_state=42, class_weight='balanced'
        )
        clf_r2.fit(X_r2, y_r2)

        if VAL_CACHE.exists():
            val_pred_r2 = clf_r2.predict(X_val[:, :X_r2.shape[1]]).astype(int)
            print(f'Round 2 val Acc±1: {acc_pm1(val_pred_r2, y_val):.4f}')

        with open(CKPT_DIR / 'pseudo_lgbm_r2.pkl', 'wb') as f:
            pickle.dump(clf_r2, f)
        print('Saved: pseudo_lgbm_r2.pkl')
else:
    print('Round 2 skipped. Set RUN_ROUND2 = True to enable.')

## HITL Decision Gate

| Outcome | Decision |
|---------|----------|
| Pseudo retrain Acc±1 > baseline Acc±1 by ≥ 0.5 pp | Include pseudo model in NB10 stack |
| Pseudo retrain Acc±1 ≈ baseline (within 0.5 pp) | Skip — overhead not worth it |
| Pseudo class distribution is dominated (> 80%) by 1 class | Reduce `CONF_THRESHOLD` to 0.90 or 0.80 |

**Next:** `10_final_stack.ipynb` — combine all models, grid-search weights, apply D4 TTA
and median filter, generate the final submission ZIP.